In [ ]:
# Segmentation is where we take photos and we colour in every pixel according to what it is.

In [ ]:
# In fastai, SegmentationDataLoaders is a high-level factory class designed to quickly load image segmentation datasets (where pixel-level masks correspond to input images).

In [18]:
import multiprocessing as mp

mp.set_start_method('fork', force=True)

In [4]:
from fastai.vision.all import *

In [6]:
path = untar_data(URLs.CAMVID_TINY)

<div><progress max="2314212" value="2318336"></progress> 100.18% [2318336/2314212 00:14&lt;00:00]</div>

In [ ]:
# The untar_data function in the fastai library safely downloads a compressed dataset URL, extracts it into your ~/.fastai/data/ directory, and returns a pathlib Path object pointing to the extracted files. It prevents re-downloading if the files already exist

In [ ]:
# In course_nb.ipynb we used data blocks which are a kind of intermediate level, very flexible
# approach that can be taken to handle amost any kind of data. But for the kinds of data that occur
# a lot, we can use special data loaders claseses which kind of let us use even less code.

In [12]:
# label_func gets the paths 
# codes gets the labels for further segmentation

# • path: The root directory where your dataset is stored (e.g., downloaded or unzipped via untar_data). All other relative paths look up from here.
# • bs=8: The batch size (bs). It tells the dataloader to process 8 image-mask pairs at a time during training.
# • fnames=get_image_files(path/"images"): A list of file paths for all input images, retrieved by searching inside the path/images folder using fastai's get_image_files helper.
# • label_func=lambda o: path/'labels'/f'{o.stem}_P{o.suffix}': A function (written as a Python lambda) that maps each input image file (o) to its corresponding segmentation mask file. For example, if an image is named foo.png, it looks for a mask in path/labels/foo_P.png.
# • codes=np.loadtxt(path/'codes.txt', dtype=str): Loads class names or category labels (such as road, car, pedestrian) from a text file (codes.txt) into a NumPy array of strings, mapping pixel mask index values to human-readable class names.

dls = SegmentationDataLoaders.from_label_func(
    path, bs=8, fnames=get_image_files(path/"images"),
    label_func=lambda o: path/'labels'/f'{o.stem}_P{o.suffix}',
    codes=np.loadtxt(path/'codes.txt', dtype=str))

What bs=8 does
bs=8 is the number of image–mask pairs the model sees in each training step. Changing it changes how the training runs.

The dataset has 100 images. fastai splits them roughly 80/20, so about 80 are used for training. With bs=8 that's about 10 gradient updates per epoch, and 8 epochs means roughly 80 updates in total.

Increasing it (e.g. 16 or 32):
Each update averages the gradient over more images, so it's less noisy.
There are fewer updates per epoch, so you may need more epochs or a higher learning rate to reach the same loss.
It uses more GPU memory, and a batch that's too large can cause out-of-memory errors.
Throughput per image usually improves, so epochs run faster.
Decreasing it (e.g. 4 or 2):
Updates are noisier, which can act as mild regularization but may make the loss jumpy.
It needs less memory.
With very small batches, the BatchNorm statistics in ResNet-34 become unreliable, which can hurt results.
Your valid_loss values would shift with bs as well, because the number of optimizer steps changes. Comparing runs with different batch sizes is only fair if you account for that.

How the image is mapped to its mask
The mapping happens in this line:
label_func=lambda o: path/'labels'/f'{o.stem}_P{o.suffix}'

For each image path o, it builds the mask path by taking the filename stem, appending _P, and keeping the same extension. The pairing is by filename, not by position:

images/0001TP_006750.png → labels/0001TP_006750_P.png

Where the mask files come from
They are not generated. They ship with the CamVid dataset. The line
path = untar_data(URLs.CAMVID_TINY)

downloads a compressed archive from fastai's dataset URLs and extracts it to ~/.fastai/data/camvid_tiny/. That folder contains:
images/: the 100 input photos
labels/: the 100 pixel-level mask PNGs, with the _P suffix
codes.txt: the class names (Animal, Archway, Bicyclist, …). Each pixel value in a mask is an index into this list.
The masks were produced by people labeling every pixel of each street-scene photo. That labeling is the ground truth the U-Net learns from.

In [ ]:
# • learn = unet_learner(dls, resnet34): Creates a U-Net architecture for image segmentation using a pre-trained ResNet-34 backbone (from the fastai Vision Learner Documentation), trained on the data loaders stored in dls.
# • learn.fine_tune(8): Trains the model for 8 epochs using fastai's transfer learning process:
# 	1. First, it trains only the newly added head layers for 1 epoch while keeping the pre-trained ResNet-34 backbone frozen.
# 	2. Next, it unfreezes the entire model and trains all layers ("fine-tuning") with discriminative learning rates for the remaining epochs (totaling 8 epochs).

In [16]:
# Fix for an error - Python is trying to start worker processes for your fastai data loaders, but it cannot serialize a lambda function used in your data transformations.

# Disable multiprocessing in fastai data loaders
# dls.train.num_workers = 0
# dls.valid.num_workers = 0

# Actually, the above does not work because fastai version is using num_workers in a calculation that divides by this value, so setting it to zero triggers ZeroDivisionError.

# dls.train.num_workers = 1
# dls.valid.num_workers = 1

# The above didn't work either, instead we tell Python to use the Linux fork method (code at start) instead of forkserver, which may avoid the lambda-pickling problem.

In [19]:
learn = unet_learner(dls, resnet34)
learn.fine_tune(8)

epoch,train_loss,valid_loss,time
0,3.404906,2.485197,00:19


epoch,train_loss,valid_loss,time
0,1.951054,1.684497,00:21
1,1.620789,1.307211,00:21
2,1.415220,1.158347,00:22
3,1.258623,0.978744,00:21
4,1.139877,0.928695,00:21
5,1.031438,0.867466,00:22
6,0.938226,0.846292,00:22
7,0.863818,0.837724,00:21


In [20]:
path

Path('/home/vaishnavi-asawale/.fastai/data/camvid_tiny')

In [ ]:
# Tabular analysis is the most widely used alongside data like that from spreadsheets

In [4]:
from fastai.tabular.all import *

In [5]:
# untar_data downloads some data and decompresses it
# There are a lot of URLs provided by fastai for all kinds of common data sets that can be used

path = untar_data(URLs.ADULT_SAMPLE)

<div><progress max="968212" value="974848"></progress> 100.69% [974848/968212 00:03&lt;00:00]</div>

In [6]:
# We need to tell TabularDataLoaders which of the columns are categorical (cat_names), and 
# which ones are continuos (cont_names)(can take any real number)

dls = TabularDataLoaders.from_csv(path/'adult.csv', path=path, y_names="salary",
                                  cat_names = ['workclass','education','marital-status','occupation','relationship'],
                                  cont_names = ['age', 'fnlwgt', 'education-num'],
                                  procs = [Categorify, FillMissing, Normalize])

dls.show_batch()

/home/vaishnavi-asawale/.local/lib/python3.14/site-packages/fastai/torch_core.py:155: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  else as_tensor(x.values, **kwargs) if isinstance(x, (pd.Series, pd.DataFrame))


,workclass,education,marital-status,occupation,relationship,education-num_na,age,fnlwgt,education-num,salary
0,?,11th,Never-married,?,Unmarried,False,18.000000,340117.002692,7.0,<50k
1,Private,Assoc-acdm,Never-married,Adm-clerical,Own-child,False,23.000001,188545.000014,12.0,<50k
2,Private,Bachelors,Divorced,Exec-managerial,Not-in-family,False,40.000000,216237.000785,13.0,>=50k
3,Self-emp-inc,HS-grad,Married-civ-spouse,Farming-fishing,Husband,False,48.000000,38240.004601,9.0,>=50k
4,Federal-gov,Some-college,Married-civ-spouse,Adm-clerical,Wife,False,55.000000,31728.004986,10.0,>=50k
5,Private,HS-grad,Never-married,Other-service,Unmarried,False,34.000000,331125.994084,9.0,<50k
6,Private,Masters,Married-civ-spouse,Prof-specialty,Husband,False,28.000000,173944.000170,14.0,>=50k
7,Private,HS-grad,Married-civ-spouse,Sales,Wife,False,39.000000,343402.999379,9.0,<50k
8,Self-emp-not-inc,Bachelors,Divorced,Craft-repair,Not-in-family,False,31.000000,144948.999923,13.0,<50k
9,State-gov,Assoc-voc,Married-civ-spouse,Exec-managerial,Husband,False,41.000000,363590.999946,11.0,<50k


In [7]:
# Fastai has something called "type dispatch" which is a system that's particularly popluar in a
# language called Julia to basically automatically, do the right thing for your data regardless of what
# kind of data it is. So if you call show_batch(), we should get batch something "useful"

In [8]:
learn = tabular_learner(dls, metrics=accuracy)
learn.fit_one_cycle(2)
# We don't call fine_tune, we use fit
# That's because for tabular data there won't be any pre-trained model for that sort of data
# Because every table of data is very different, whereas pictures have the same sort of "idea"

epoch,train_loss,valid_loss,accuracy,time
0,0.366049,0.362535,0.830160,00:04
1,0.354472,0.351474,0.833692,00:04


In [ ]:
# Collaborative filtering - the foundation of recommendation systems
# It's a system where we basically take a data set that says which users liked/ used which 
# products, and then we use that to guess what other producs those users might like based on finding 
# similar users, and what those similar users like

In [10]:
from fastai.collab import *

path = untar_data(URLs.ML_SAMPLE)
dls = CollabDataLoaders.from_csv(path/'ratings.csv')

<div><progress max="51790" value="57344"></progress> 110.72% [57344/51790 00:00&lt;00:00]</div>

In [11]:
dls.show_batch()

,userId,movieId,rating
0,423,592,2.5
1,574,733,3.0
2,130,527,4.0
3,624,1036,5.0
4,460,2959,3.5
5,480,2959,3.5
6,17,1213,5.0
7,654,364,5.0
8,311,588,3.0
9,505,608,3.0


In [13]:
# Collaborative data has some kind of user_id, some product_id (movie), and a rating

# Since this is not predicting a cateogry but a real number, we tell it the possible range (y_range)
# The actual range is 1-5, but it is a good idea to add some padding (0.5 - 5.5)
learn = collab_learner(dls, y_range=(0.5, 5.5))
learn.fine_tune(10) # fit can also be used

epoch,train_loss,valid_loss,time
0,1.496672,1.369415,00:00


epoch,train_loss,valid_loss,time
0,1.397873,1.317113,00:00
1,1.285793,1.165504,00:01
2,1.058627,0.879739,00:01
3,0.822068,0.714031,00:01
4,0.703935,0.675692,00:00
5,0.659686,0.666682,00:00
6,0.630955,0.659062,00:00
7,0.624529,0.655287,00:01
8,0.597424,0.654959,00:01
9,0.614137,0.654828,00:00


In [14]:
# Valid loss is the mean squared error, and it goes down each epoch

learn.show_results()

,userId,movieId,rating,rating_pred
0,13.0,69.0,5.0,3.985498
1,75.0,54.0,3.0,4.657234
2,89.0,52.0,3.5,4.005715
3,61.0,79.0,3.0,3.407192
4,11.0,84.0,4.0,3.623132
5,11.0,67.0,3.0,3.626521
6,48.0,75.0,4.0,4.080445
7,16.0,7.0,4.0,4.392134
8,98.0,47.0,4.5,4.796089
